In [1]:
from contextlib import contextmanager
import lseg.data as ld
# import matplotlib.pyplot as plt
# import matplotlib.dates as mdates
from lseg.data.discovery import Chain
from IPython.display import display, clear_output
# import seaborn as sns
from datetime import date
import pandas as pd
from functools import reduce

# end_date = (datetime.now() - timedelta(weeks=1)).strftime('%Y-%m-%d')
pd.set_option('future.no_silent_downcasting', True)

@contextmanager
def ld_session():
    """Context manager for handling a LSEG Data Library for Python session."""
    try:
        # Create a session and return it
        ld.open_session()
        yield
    finally:
        # Close the session
        ld.close_session()
        
today_str = date.today().strftime('%Y-%m-%d')

In [4]:
# Initialize dictionary
mena_banks_by_country = {}

with ld.open_session():
    # 1. Get the top-level constituents (the country-specific chains)
    main_chain = Chain(name="0#.TRXFLDGCPUBANK")
    country_chains = main_chain.constituents
    country_chains.append('0#.TRXFLDSAPBANK') # add Saudis manually
    # 2. Loop through and pull constituents for each country chain separately
    for tic in country_chains[1:]:
        # Create a dictionary entry for each country's sub-chain
        sub_chain = Chain(name=tic)
        mena_banks_by_country[tic] = sub_chain.constituents

display(country_chains)

['.TRXFLDGCPUBANK',
 '0#.TRXFLDAEPBANK',
 '0#.TRXFLDBHPBANK',
 '0#.TRXFLDQAPBANK',
 '0#.TRXFLDOMPBANK',
 '0#.TRXFLDKWPBANK',
 '0#.TRXFLDSAPBANK']

In [5]:
all_country_dfs = []

with ld.open_session():
    for country_ric, tickers in mena_banks_by_country.items():
        if not tickers:
            continue
            
        print(f"Pulling data for {country_ric}...")
        
        # Pulling country-specific data
        df = ld.get_history(
            universe=tickers, 
            fields=['TR.F.TotAssets', 'TR.F.ComEqTot', 'TR.F.TotLiab', 'TR.F.TotLiabEq'],
            parameters={
                'Curn': 'Native', 
                'Period': 'FQ0', 
                'Frq': 'FQ',      # Quarterly frequency for fundamentals
                'SDate': '0', 
                'EDate': '2010-01-01'   
            }
        )
        
        # Optional: Add a column to identify the source country chain
        df['Country_Chain'] = country_ric
        all_country_dfs.append(df)

# To get the intersection of all countries (only dates where all have reported)
intersected_fundamentals = reduce(lambda left, right: left.join(right, how='inner', rsuffix='_drop'), all_country_dfs)
display(intersected_fundamentals.head())

Pulling data for 0#.TRXFLDAEPBANK...
Pulling data for 0#.TRXFLDBHPBANK...
Pulling data for 0#.TRXFLDQAPBANK...
Pulling data for 0#.TRXFLDOMPBANK...
Pulling data for 0#.TRXFLDKWPBANK...
Pulling data for 0#.TRXFLDSAPBANK...


AJBNK.DU                                          \
           Total Assets Common Equity - Total Total Liabilities   
Date                                                              
2009-12-31         <NA>                  <NA>              <NA>   
2010-03-31   2169968000             989143000        1180825000   
2010-06-30   2595447000             987101000        1608346000   
2010-09-30   2923798000             983821000        1939977000   
2010-12-31   3234346000            1006942000        2227404000   

                                           DISB.DU                        \
           Total Liabilities & Equity Total Assets Common Equity - Total   
Date                                                                       
2009-12-31                       <NA>  84304271000            8975890000   
2010-03-31                 2169968000  84971466000            8712483000   
2010-06-30                 2595447000  83803879000            8857855000   
2010-09-30                 2923798000  81847502000            9223282000   
2010-12-31                 3234346000  89884401000            9326079000   

                                                              ENBD.DU  \
           Total Liabilities Total Liabilities & Equity  Total Assets   
Date                                                                    
2009-12-31       75323471000                84304271000  281576482000   
2010-03-31       76258160000                84971466000  289723434000   
2010-06-30       74944285000                83803879000  282340152000   
2010-09-30       72621737000                81847502000  284222120000   
2010-12-31       79611314000                89884401000  286078324000   

                                  ...                    1030.SE      1183.SE  \
           Common Equity - Total  ... Total Liabilities & Equity Total Assets   
Date                              ...                                           
2009-12-31           27876499000  ...                50148011000         <NA>   
2010-03-31           29539636000  ...                49491858000         <NA>   
2010-06-30           28686491000  ...                48866204000         <NA>   
2010-09-30           28667097000  ...                49315982000         <NA>   
2010-12-31           29655770000  ...                51491233000         <NA>   

                                                                               \
           Common Equity - Total Total Liabilities Total Liabilities & Equity   
Date                                                                            
2009-12-31                  <NA>              <NA>                       <NA>   
2010-03-31                  <NA>              <NA>                       <NA>   
2010-06-30                  <NA>              <NA>                       <NA>   
2010-09-30                  <NA>              <NA>                       <NA>   
2010-12-31                  <NA>              <NA>                       <NA>   

                 1180.SE                                          \
            Total Assets Common Equity - Total Total Liabilities   
Date                                                               
2009-12-31  257452175000           29271087000      226592016000   
2010-03-31  264823198000           28823548000      234401412000   
2010-06-30  263271997000           29114378000      232599895000   
2010-09-30  271232514000           30873860000      238704891000   
2010-12-31  282371992000           31272258000      249515299000   

                                      Country_Chain_drop  
           Total Liabilities & Equity                     
Date                                                      
2009-12-31               257452175000   0#.TRXFLDSAPBANK  
2010-03-31               264823198000   0#.TRXFLDSAPBANK  
2010-06-30               263271997000   0#.TRXFLDSAPBANK  
2010-09-30               271232514000   0#.TRXFLDSAPBANK  
2010-12-31               282

In [8]:
# holidays

with ld.open_session():
    holidays = ld.dates_and_calendars.holidays(
    start_date="2016-01-01", # cannot set earlier! I guess we might need to start somewhere here if we want to cauciously remove holidays
    end_date=today_str,
    calendars=["UAE", "KWT", "SAU", "OMN", "BAH"])
 
holidays.df

,name,calendars,countries,tag,date
0,New Year's Day,"[UAE, OMN, BAH]","[ARE, OMN, BHS]",None,2016-01-01
1,New Years Day,[KWT],[KWT],None,2016-01-01
2,Prophets Ascension '16,[UAE],[ARE],None,2016-05-05
3,Prophet Mohammeds Ascension Day '16,[KWT],[KWT],None,2016-05-05
4,Prophet Mohammed's Ascension Day (Lailat al Mi...,[OMN],[OMN],None,2016-05-05
...,...,...,...,...,...
676,National Heroes Day (Observed),[BAH],[BHS],None,2025-10-13
677,Christmas Day,[BAH],[BHS],None,2025-12-25
678,Boxing Day,[BAH],[BHS],None,2025-12-26
679,Majority Rule Day,[BAH],[BHS],None,2026-01-10


In [9]:
# These are Banking-sector benchmarks
# This creates a new list with '0#' gone
benchmarksRic = list(map(lambda ric: ric.replace('0#', ''), country_chains))

with ld.open_session():
    benchm = ld.get_history(
                universe=benchmarksRic, 
                fields='TR.PriceClose', # TR.PriceClose field always returns adjusted stock prices.
                parameters={'Curn': 'Native', 'Fill': 'None'},
                start='2009-12-31', 
                end=today_str, 
                interval='daily')
    benchInf = ld.get_data(universe=benchmarksRic, fields=["TR.IndexGeography","TR.IndexSeriesName"])
    

display(benchm.head())
display(benchInf)

Price Close,.TRXFLDGCPUBANK,.TRXFLDAEPBANK,.TRXFLDBHPBANK,.TRXFLDQAPBANK,.TRXFLDOMPBANK,.TRXFLDKWPBANK,.TRXFLDSAPBANK
Date,,,,,,,
2010-09-16,199.1479,200.74021,68.2576,200.91137,145.94359,207.66798,<NA>
2010-09-17,199.1479,200.74021,68.2576,200.91137,145.94359,207.66798,<NA>
2010-09-20,200.67,204.59,68.26,202.4,148.32,207.47,<NA>
2010-09-21,200.67,204.57,68.26,203.76,147.84,205.99,<NA>
2010-09-22,201.09,203.57,68.61,203.68,147.31,207.33,<NA>


,Instrument,Index Geography,Index Series Name
0,.TRXFLDGCPUBANK,Middle East,TR Indices EMA Series
1,.TRXFLDAEPBANK,United Arab Emirates,TR Indices EMA Series
2,.TRXFLDBHPBANK,Bahrain,TR Indices EMA Series
3,.TRXFLDQAPBANK,Qatar,TR Indices EMA Series
4,.TRXFLDOMPBANK,Oman,TR Indices EMA Series
5,.TRXFLDKWPBANK,Kuwait,TR Indices EMA Series
6,.TRXFLDSAPBANK,Saudi Arabia,TR Indices EMA Series


In [9]:
# here is TR Indices EMA Series (GCC PRice return index) and S&P GCC COMPOSITE (preferred?)
benchmarkMainRic = ['.TRXFLDGCPU', '.GPDGC']

with ld.open_session():
    benchmMain = ld.get_history(
                universe=benchmarkMainRic, 
                fields='TR.PriceClose',
                parameters={'Curn': 'Native', 'Fill': 'None'},
                start='2009-12-31', 
                end=today_str, 
                interval='daily')
    benchMainInf = ld.get_data(universe=benchmarkMainRic, fields=["TR.IndexGeography","TR.IndexSeriesName"])

display(benchmMain.head())
display(benchMainInf)

Price Close,.TRXFLDGCPU,.GPDGC
Date,,
2009-12-31,<NA>,88.7583
2010-01-04,<NA>,89.1077
2010-01-05,<NA>,89.8954
2010-01-06,<NA>,90.1913
2010-01-07,<NA>,90.4951


,Instrument,Index Geography,Index Series Name
0,.TRXFLDGCPU,Middle East,TR Indices EMA Series
1,.GPDGC,,S&P Equity Index Series


In [10]:
country_data_frames = {}
fields_to_pull = ['TR.TotalReturn', 'TR.PriceClose', 'TR.CompanyMarketCapitalization']

with ld.open_session():
    for country_ric, tickers in mena_banks_by_country.items():
        if not tickers: continue
        
        field_frames = []
        for field in fields_to_pull:
            df_field = ld.get_history(
                universe=tickers, 
                fields=[field],
                parameters={'Curn': 'Native', 'Fill': 'None'},
                start='2009-12-31', 
                end=today_str, 
                interval='daily'
            )
            
            if not df_field.empty:
                # 1. Clean duplicates for this specific field
                df_field = df_field.groupby(df_field.index).first()
                
                # 2. FORCE MULTIINDEX: Create (RIC, Field) tuples for columns
                # This ensures every column is explicitly labeled
                new_columns = pd.MultiIndex.from_product([df_field.columns, [field]], names=['RIC', 'Field'])
                df_field.columns = new_columns
                
                field_frames.append(df_field)
        
        if field_frames:
            # Join fields for this country
            country_df = pd.concat(field_frames, axis=1)
            country_data_frames[country_ric] = country_df

# Final Intersection (Inner Join)
intersected_eq_data = pd.concat(country_data_frames.values(), axis=1, join='inner')

# NOW this will work beautifully:
display(intersected_eq_data['ENBD.DU'].head())

KeyboardInterrupt: 